<button data-commandLinker-command="progressivis:cleanup_and_run" data-commandlinker-args='{"index": 1}' href="#" class='progressivis-cleanup-and-run-btn'>Run ProgressiVis</button>

In [1]:
from ipyprogressivis.widgets.chaining.constructor import Constructor
from ipyprogressivis.widgets.chaining.utils import create_root, get_header
from ipyprogressivis.widgets.chaining.custom import *
# ***************************************************************************************
# WARNING: This cell must only be executed using the 'Run ProgressiVis' button above.
# Do not execute it in any other way, as the result will not be as expected.
# For the same reason do not copy/paste the contents of this cell to execute it elsewhere
# ***************************************************************************************
header = get_header()
display(header.talker)
display(header.backup)
_ = header.constructor
with header.modules_out:
    display(header.board)
with header.widgets_out:
    display(header.manager)
header.talker.labcommand("notebook:hide-cell-code")
%reload_ext ipyprogressivis.magics
create_root(header.backup)

Talker()

BackupWidget()

## root

In [2]:
# do not run this cell
display(header.constructor)
header.constructor.start_scheduler()
header.talker.labcommand('notebook:hide-cell-code')

Constructor(children=(IntProgress(value=0, description='Starting ProgressiVis ...', max=2, style=ProgressStyle…

Starting scheduler
# Scheduler added module(s): ['sink_1', 'variable_1']
# Scheduler added module(s): ['constant_1', 'simple_csv_loader_1', 'sink_2']


## CSV loader

In [3]:
Constructor.widget('CSV loader', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

## Objective
In this notebook, we aim to identify specific days of the week and time intervals during the day when incident volumes reach their peak. These insights will help us strategically allocate resolution groups to improve incident response efficiency and strengthen our overall handling capacity. Additionally, we will examine how incident volume influences the likelihood of SLA breaches.

## Snippets

### Distribution of unique incidents by weekday

In [4]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
from datetime import datetime

@register_snippet
def progressive_weekday_boxplot(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_weekday_boxplot, "_state"):

        # ----------------------------
        # Progress bar + unique count
        # ----------------------------
        progress_bar = IntProgress(value=0, min=0, max=100,
                                   description='Streaming:',
                                   bar_style='info',
                                   layout={'width':'70%'})
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Box plot for weekdays
        # ----------------------------
        box_fig = go.FigureWidget()
        # Initialize empty boxes for each weekday
        weekdays = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'cyan']
        for wd, color in zip(weekdays, colors):
            box_fig.add_trace(go.Box(
                y=[],
                name=wd,
                marker_color=color,
                boxpoints=False  # hide individual points / blue dots
            ))
        box_fig.update_layout(
            title="Distribution of Unique Incidents by Weekday (First Occurrence)",
            xaxis_title="Weekday",
            yaxis_title="Unique Incidents Count",
            width=900,
            height=500,
            margin=dict(l=80, r=40, t=80, b=120)
        )

        # ----------------------------
        # Persistent state
        # ----------------------------
        state = {
            "cursor": 0,
            "batch_size": 50,
            "seen_incidents": {},  # incident number -> first occurrence weekday
            "unique_incidents": set(),
            "box_fig": box_fig,
            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "weekdays": weekdays,
            "weekday_data": {wd: [] for wd in weekdays}  # values for each box
        }

        progressive_weekday_boxplot._state = state

        with out:
            display(VBox([top_bar, box_fig]))

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:
                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        if incident_id in state["seen_incidents"]:
                            continue  # only consider first occurrence

                        opened_at = row.get("opened_at")
                        if opened_at is None:
                            continue
                        # Parse date, assuming format like "29/2/2016 01:16"
                        try:
                            dt = datetime.strptime(opened_at, "%d/%m/%Y %H:%M")
                        except:
                            continue
                        weekday = dt.strftime("%A")  # full weekday name

                        if weekday not in state["weekdays"]:
                            continue  # skip invalid

                        # Track first occurrence
                        state["seen_incidents"][incident_id] = weekday
                        state["weekday_data"][weekday].append(len(state['weekday_data'][weekday])+1)  # count 1 per incident
                        state["unique_incidents"].add(incident_id)

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"] / table.nrow) * 100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = f"Unique Incidents: {len(state['unique_incidents'])}"

                    # ----------------------------
                    # Update figure
                    # ----------------------------
                    for idx, wd in enumerate(state["weekdays"]):
                        state["box_fig"].data[idx].y = state["weekday_data"][wd]

                await aio.sleep(0.0001)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Running average of incidents per hour

In [6]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import datetime
import math

@register_snippet
def progressive_hourly_running_avg(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_hourly_running_avg, "_state"):

        # ----------------------------
        # Progress bar + label
        # ----------------------------
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Line chart + CI band
        # ----------------------------
        fig = go.FigureWidget()

        hours = list(range(24))
        fig.add_trace(go.Scatter(
            x=hours, y=[0]*24,
            mode='lines+markers',
            name='Running Avg',
            line=dict(color='blue'),
            marker=dict(size=6),
            hovertemplate="Hour: %{x}<br>Running Avg: %{y:.2f}<extra></extra>"
        ))

        # Shaded CI band (upper + lower)
        fig.add_trace(go.Scatter(
            x=hours + hours[::-1],
            y=[0]*24 + [0]*24,
            fill='toself',
            fillcolor='rgba(0,0,255,0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            showlegend=True,
            name='95% CI'
        ))

        fig.update_layout(
            title="Running Average of Incidents per Hour with 95% CI",
            xaxis_title="Hour of Day",
            yaxis_title="Average Number of Incidents",
            width=900,
            height=500,
            margin=dict(l=80, r=40, t=80, b=80)
        )

        # ----------------------------
        # Persistent state
        # ----------------------------
        state = {
            "cursor": 0,
            "batch_size": 50,
            "unique_incidents": {},  # incident_id -> datetime
            "hour_running_avg": [0.0]*24,
            "hour_ci": [(0,0)]*24,  # (lower, upper) per hour
            "fig": fig,
            "progress_bar": progress_bar,
            "unique_label": unique_label
        }
        progressive_hourly_running_avg._state = state

        with out:
            display(VBox([top_bar, fig]))

        # ----------------------------
        # Helper: parse opened_at to datetime
        # ----------------------------
        def parse_opened_at(val):
            if isinstance(val, str):
                try:
                    return datetime.datetime.strptime(val, "%d/%m/%Y %H:%M")
                except:
                    return None
            return val if isinstance(val, datetime.datetime) else None

        # ----------------------------
        # Streaming processing loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:
                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        opened_at_raw = row.get("opened_at")
                        dt = parse_opened_at(opened_at_raw)
                        if dt is None:
                            continue

                        # store earliest occurrence
                        if incident_id not in state["unique_incidents"]:
                            state["unique_incidents"][incident_id] = dt
                        else:
                            if dt < state["unique_incidents"][incident_id]:
                                state["unique_incidents"][incident_id] = dt

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"]/table.nrow)*100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = f"Unique Incidents: {len(state['unique_incidents'])}"

                    # ----------------------------
                    # Aggregate counts per day/hour
                    # ----------------------------
                    day_hour_counts = defaultdict(lambda: [0]*24)
                    for dt in state["unique_incidents"].values():
                        day = dt.date()
                        hour = dt.hour
                        day_hour_counts[day][hour] += 1

                    # ----------------------------
                    # Compute running avg + CI per hour
                    # ----------------------------
                    hour_avg = []
                    hour_lower = []
                    hour_upper = []

                    for h in range(24):
                        # values for this hour across days
                        values = [counts[h] for counts in day_hour_counts.values()]
                        n = len(values)
                        mean = sum(values)/n if n>0 else 0.0
                        # running standard deviation
                        if n>1:
                            var = sum((v - mean)**2 for v in values)/(n-1)
                            se = math.sqrt(var/n)
                        else:
                            se = 0.0
                        ci = 1.96 * se
                        hour_avg.append(mean)
                        hour_lower.append(mean - ci)
                        hour_upper.append(mean + ci)

                    state["hour_running_avg"] = hour_avg
                    state["hour_ci"] = list(zip(hour_lower, hour_upper))

                    # ----------------------------
                    # Update figure
                    # ----------------------------
                    # line
                    state["fig"].data[0].y = hour_avg
                    # CI band
                    state["fig"].data[1].y = hour_upper + hour_lower[::-1]

                await aio.sleep(0.05)

        aio.create_task(process_rows())

    return SnippetResult(output_module=input_module, output_slot=input_slot, widget=out)

### Unique incidents per hour by weekday

In [13]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import datetime
import re

@register_snippet
def progressive_hourly_weekday_line(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_hourly_weekday_line, "_state"):

        # ----------------------------
        # Progress bar
        # ----------------------------
        progress_bar = IntProgress(value=0, min=0, max=100,
                                   description='Streaming:',
                                   bar_style='info',
                                   layout={'width':'70%'})
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Line chart for hourly incidents by weekday
        # ----------------------------
        fig = go.FigureWidget()
        weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'cyan']

        weekday_data = {day: [0]*24 for day in weekdays}  # cumulative count per hour

        for idx, day in enumerate(weekdays):
            fig.add_trace(go.Scatter(
                x=list(range(24)),
                y=weekday_data[day],
                mode='lines+markers',
                name=day,
                line=dict(color=colors[idx]),
                marker=dict(size=6)
            ))

        fig.update_layout(
            title="Unique Incidents per Hour by Weekday",
            xaxis_title="Hour of Day",
            yaxis_title="Unique Incident Count",
            width=1000,
            height=500,
            margin=dict(l=80, r=40, t=80, b=80)
        )

        # ----------------------------
        # State
        # ----------------------------
        state = {
            "cursor": 0,
            "batch_size": 50,
            "seen_incidents": {},  # store first occurrence of each incident
            "unique_incidents": set(),
            "fig": fig,
            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "weekday_data": weekday_data
        }

        progressive_hourly_weekday_line._state = state

        with out:
            display(VBox([top_bar, fig]))

        # ----------------------------
        # Streaming processing
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:
                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        if incident_id in state["unique_incidents"]:
                            continue  # only first occurrence

                        state["unique_incidents"].add(incident_id)

                        opened_at_raw = row.get("opened_at", "")
                        try:
                            # Example: "29/02/2016 01:16"
                            dt = datetime.datetime.strptime(opened_at_raw, "%d/%m/%Y %H:%M")
                            hour = dt.hour
                            weekday = dt.strftime("%A")  # Monday, Tuesday...
                        except:
                            continue

                        if weekday not in state["weekday_data"]:
                            continue

                        state["weekday_data"][weekday][hour] += 1

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress bar
                    # ----------------------------
                    progress_pct = int((state["cursor"] / table.nrow) * 100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = f"Unique Incidents: {len(state['unique_incidents'])}"

                    # ----------------------------
                    # Update figure
                    # ----------------------------
                    for idx, day in enumerate(weekdays):
                        state["fig"].data[idx].y = state["weekday_data"][day]

                await aio.sleep(0.01)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### Incident categories at time interval in a day

In [15]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import datetime

@register_snippet
def progressive_category_by_time(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_category_by_time, "_state"):

        # ----------------------------
        # Progress bar
        # ----------------------------
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # Morning chart
        # ----------------------------
        morning_fig = go.FigureWidget()
        morning_fig.add_trace(go.Bar(
            x=[], y=[],
            marker_color='dodgerblue',
            hovertemplate="Category: %{x}<br>Count: %{y}<extra></extra>"
        ))
        morning_fig.update_layout(
            title="Incident Categories: Morning (8-11)",
            xaxis_title="Category",
            yaxis_title="Number of Incidents",
            width=1200,
            height=500,
            margin=dict(l=80, r=40, t=80, b=160)
        )

        # ----------------------------
        # Afternoon chart
        # ----------------------------
        afternoon_fig = go.FigureWidget()
        afternoon_fig.add_trace(go.Bar(
            x=[], y=[],
            marker_color='orange',
            hovertemplate="Category: %{x}<br>Count: %{y}<extra></extra>"
        ))
        afternoon_fig.update_layout(
            title="Incident Categories: Afternoon (13-16)",
            xaxis_title="Category",
            yaxis_title="Number of Incidents",
            width=1200,
            height=500,
            margin=dict(l=80, r=40, t=80, b=160)
        )

        # ----------------------------
        # State
        # ----------------------------
        state = {
            "cursor": 0,
            "batch_size": 50,
            "unique_incidents": {},
            "morning_counts": defaultdict(int),
            "afternoon_counts": defaultdict(int),
            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "morning_fig": morning_fig,
            "afternoon_fig": afternoon_fig
        }

        progressive_category_by_time._state = state

        with out:
            display(VBox([top_bar, morning_fig, afternoon_fig]))

        # ----------------------------
        # Helper
        # ----------------------------
        def parse_opened_at(val):
            if isinstance(val, str):
                try:
                    return datetime.datetime.strptime(val, "%d/%m/%Y %H:%M")
                except:
                    return None
            return val if isinstance(val, datetime.datetime) else None

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:

                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None:
                            continue

                        opened_at_raw = row.get("opened_at")
                        dt = parse_opened_at(opened_at_raw)
                        if dt is None:
                            continue

                        hour = dt.hour

                        # Count only first occurrence
                        if incident_id not in state["unique_incidents"]:
                            state["unique_incidents"][incident_id] = row
                            category = row.get("category", "Unknown")

                            if 8 <= hour <= 11:
                                state["morning_counts"][category] += 1
                            elif 13 <= hour <= 16:
                                state["afternoon_counts"][category] += 1

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"] / table.nrow) * 100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = (
                        f"Unique Incidents: {len(state['unique_incidents'])}"
                    )

                    # ----------------------------
                    # SORT + Update Morning Chart
                    # ----------------------------
                    sorted_morning = sorted(
                        state["morning_counts"].items(),
                        key=lambda x: x[1],
                        reverse=True
                    )

                    morning_x = [item[0] for item in sorted_morning]
                    morning_y = [item[1] for item in sorted_morning]

                    state["morning_fig"].data[0].x = morning_x
                    state["morning_fig"].data[0].y = morning_y

                    # ----------------------------
                    # SORT + Update Afternoon Chart
                    # ----------------------------
                    sorted_afternoon = sorted(
                        state["afternoon_counts"].items(),
                        key=lambda x: x[1],
                        reverse=True
                    )

                    afternoon_x = [item[0] for item in sorted_afternoon]
                    afternoon_y = [item[1] for item in sorted_afternoon]

                    state["afternoon_fig"].data[0].x = afternoon_x
                    state["afternoon_fig"].data[0].y = afternoon_y

                await aio.sleep(0.05)

        aio.create_task(process_rows())

    return SnippetResult(
        output_module=input_module,
        output_slot=input_slot,
        widget=out
    )

### SLA breach by time slot

In [17]:
# progressivis-snippet
from ipyprogressivis.widgets.chaining.custom import register_snippet, SnippetResult
from ipywidgets import Output, VBox, HBox, IntProgress, Label
import plotly.graph_objects as go
import progressivis.core.aio as aio
from collections import defaultdict
import datetime

@register_snippet
def progressive_sla_breach_timeslot(input_module, input_slot, columns):
    table = input_module.output[input_slot].data()
    out = Output()

    if not hasattr(progressive_sla_breach_timeslot, "_state"):

        # ----------------------------
        # Progress bar
        # ----------------------------
        progress_bar = IntProgress(
            value=0, min=0, max=100,
            description='Streaming:',
            bar_style='info',
            layout={'width':'70%'}
        )
        unique_label = Label("Unique Incidents: 0")
        top_bar = HBox([progress_bar, unique_label])

        # ----------------------------
        # SLA Breach Bar Chart
        # ----------------------------
        fig = go.FigureWidget()
        fig.add_trace(go.Bar(
            x=["8-11", "13-16"],
            y=[0,0],
            marker_color='crimson',
            hovertemplate="Time Slot: %{x}<br>Breached Count: %{y}<extra></extra>"
        ))
        fig.update_layout(
            title="SLA Breaches by Time Slot",
            xaxis_title="Time Slot",
            yaxis_title="Number of Breached SLAs",
            width=800,
            height=500,
            margin=dict(l=80, r=40, t=80, b=120)
        )

        # ----------------------------
        # Persistent State
        # ----------------------------
        state = {
            "cursor": 0,
            "batch_size": 50,
            "unique_incidents": {},  # incident_id -> row
            "breach_counts": {"8-11": 0, "13-16": 0},
            "progress_bar": progress_bar,
            "unique_label": unique_label,
            "fig": fig
        }
        progressive_sla_breach_timeslot._state = state

        with out:
            display(VBox([top_bar, fig]))

        # ----------------------------
        # Helper
        # ----------------------------
        def parse_opened_at(val):
            if isinstance(val, str):
                try:
                    return datetime.datetime.strptime(val, "%d/%m/%Y %H:%M")
                except:
                    return None
            return val if isinstance(val, datetime.datetime) else None

        # ----------------------------
        # Streaming loop
        # ----------------------------
        async def process_rows():
            while True:
                if state["cursor"] < table.nrow:
                    start = state["cursor"]
                    end = min(start + state["batch_size"], table.nrow)

                    for i in range(start, end):
                        row = table.row(i)
                        incident_id = row.get("number")
                        if incident_id is None or incident_id in state["unique_incidents"]:
                            continue

                        opened_at_raw = row.get("opened_at")
                        dt = parse_opened_at(opened_at_raw)
                        if dt is None:
                            continue
                        hour = dt.hour

                        # Track only first occurrence
                        state["unique_incidents"][incident_id] = row

                        sla_raw = row.get("made_sla", False)
                        breached = sla_raw in [True, "TRUE", "True", 1]  # SLA breached

                        if breached:
                            if 8 <= hour <= 11:
                                state["breach_counts"]["8-11"] += 1
                            elif 13 <= hour <= 16:
                                state["breach_counts"]["13-16"] += 1

                    state["cursor"] = end

                    # ----------------------------
                    # Update progress
                    # ----------------------------
                    progress_pct = int((state["cursor"] / table.nrow) * 100)
                    state["progress_bar"].value = progress_pct
                    state["unique_label"].value = f"Unique Incidents: {len(state['unique_incidents'])}"

                    # ----------------------------
                    # Update figure
                    # ----------------------------
                    state["fig"].data[0].y = [state["breach_counts"]["8-11"], state["breach_counts"]["13-16"]]

                await aio.sleep(0.05)

        aio.create_task(process_rows())

    return SnippetResult(output_module=input_module, output_slot=input_slot, widget=out)

## Distribution of unique incident by weekdays

In [5]:
Constructor.widget('Snippet', 0)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights

* **Incident volume is highest on Monday and gradually decreases as the week progresses.** This suggests that the beginning of the week experiences a surge in operational activity.

* **The lowest number of incidents occurs on Saturday and Sunday.** This likely reflects reduced business operations and lower system usage during weekends.

* **A plausible explanation for this pattern is that operational and business activities are typically higher at the start of the week.** Employees resume work, pending tasks are executed, and system interactions increase after the weekend. As the week progresses, operational intensity may stabilize or decrease, leading to fewer new incidents. Reduced activity during weekends naturally results in fewer incident reports.

Now we will look at what point of time in a day, the incident number peaks

## Running average of incident per hour

In [7]:
Constructor.widget('Snippet', 1)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights


* **Incident volumes peak between 08:00–11:00 and 13:00–16:00**, indicating that these intervals represent the highest operational activity during the day.

* **This insight can be used to strategically allocate resolution teams during these peak weekday hours**, ensuring improved response capacity and better SLA adherence.

* **The concentration of incidents during standard business hours suggests a strong relationship between user activity and incident generation.** The reduction in incident volume around midday may indicate temporary declines in system usage, possibly aligned with organizational break periods.

Now let's analyze if this is true for every weekdays 


## Unique incidents per hour by weekday

In [14]:
Constructor.widget('Snippet', 4)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights
- As we can see this is a trend from very first week, on every weekdays
- Every weekday has similar trend, where the incident count peaks during 8.00-11.00 and 13.00-16.00
- This solidifies our previous insight

Now let's check what categories of incident occurs during this time and are there any difference in categories during this time interval

## Incident categories by timeinterval

In [16]:
Constructor.widget('Snippet', 5)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights 
- category 42, 46, 53 and 26 are the most recurring in both the time intervals

## SLA breach by time slot

In [18]:
Constructor.widget('Snippet', 6)

NodeCarrier(children=(HBox(children=(Button(button_style='danger', icon='trash', style=ButtonStyle(), tooltip=…

### Insights
- As expected SLA breach by time slot, follows the pattern, most breach happens at time interval 8.00-11.00
- This can be attributed to the fact that more number of incident arrives at this time slot.
- This also suggests we need to dedicate more resource during this time slots.